In [10]:
#!/usr/bin/env python3
#coding=utf-8
import time
from Arm_Lib import Arm_Device
import numpy as np
import ikpy.chain
import ikpy.link
import ipywidgets as widgets
from IPython.display import display
#オブジェクト作成
Arm = Arm_Device()
time.sleep(.1)

In [2]:
#各サーボの角度を取得
for id in range(6):
    arm_position=Arm.Arm_serial_servo_read_any(id+1)
    print(f'{id+1} arm position {arm_position}')

1 arm position 89
2 arm position 90
3 arm position 90
4 arm position 90
5 arm position 48
6 arm position 90


In [3]:
dofbot_chain = ikpy.chain.Chain(links=[
    ikpy.link.OriginLink(), # 0: 地面
    
    ikpy.link.URDFLink(
        name="id1_base_turn",   # 1: サーボID 1 (台座)
        origin_translation=np.array([0.0, 0.0, 0.0675]), 
        origin_orientation=np.array([0.0, 0.0, 0.0]), # ★追加：初期の傾き（0度）
        rotation=np.array([0.0, 0.0, 1.0]), bounds=(np.radians(-90), np.radians(90))
    ),
    ikpy.link.URDFLink(
        name="id2_shoulder",    # 2: サーボID 2 (肩)
        origin_translation=np.array([0.0, 0.0, 0.040]),  
        origin_orientation=np.array([0.0, 0.0, 0.0]), # ★追加
        rotation=np.array([0.0, 1.0, 0.0]), bounds=(np.radians(-90), np.radians(90))
    ),
    ikpy.link.URDFLink(
        name="id3_elbow",       # 3: サーボID 3 (肘)
        origin_translation=np.array([0.0, 0.0, 0.08285]), 
        origin_orientation=np.array([0.0, 0.0, 0.0]), # ★追加
        rotation=np.array([0.0, 1.0, 0.0]), bounds=(np.radians(-80), np.radians(80))
    ),
    ikpy.link.URDFLink(
        name="id4_wrist_pitch", # 4: サーボID 4 (手首上下)
        origin_translation=np.array([0.0, 0.0, 0.08285]), 
        origin_orientation=np.array([0.0, 0.0, 0.0]), # ★追加
        rotation=np.array([0.0, 1.0, 0.0]), bounds=(np.radians(-75), np.radians(75))
    ),
    ikpy.link.URDFLink(
        name="id5_wrist_roll",  # 5: サーボID 5 (手首ひねり)
        origin_translation=np.array([0.0, 0.0, 0.07905]), 
        origin_orientation=np.array([0.0, 0.0, 0.0]), # ★追加
        rotation=np.array([0.0, 0.0, 1.0]), bounds=(np.radians(-90), np.radians(90))
    ),
    ikpy.link.URDFLink(
        name="gripper_tip",     # 6: ハサミの先端（ダミー）
        origin_translation=np.array([0.0, 0.0, 0.1203]),  
        origin_orientation=np.array([0.0, 0.0, 0.0]), # ★追加
        rotation=np.array([0.0, 0.0, 0.0]), bounds=(0, 0)
    )
], active_links_mask=[False, True, True, True, True, True, False])

In [16]:
def cal_servo_angles(target_x,target_y,target_z):
    # =====================================================================
    # 2. 動かしたい「目標の場所（座標）」を決める
    # =====================================================================
    # 例：ロボットの正面「まっすぐ前方に15cm、高さ25cm」の位置へ移動させたい場合
    target_position = np.array([target_x, target_y, target_z])
    
    # 手先の向き：ハサミを「真下」に向ける（ピッチ90度）
    pitch = np.radians(90)
    target_orientation = np.array([
    [np.cos(pitch),  0.0, np.sin(pitch)],
    [0.0,            1.0, 0.0          ],
    [-np.sin(pitch), 0.0, np.cos(pitch)]
    ])
    
    # 4x4の目標行列（ターゲットフレーム）を作成
    target_frame = np.eye(4)
    target_frame[:3, :3] = target_orientation
    target_frame[:3, 3] = target_position
    
    # =====================================================================
    # 3. 逆運動学の計算をスタート
    # =====================================================================
    # 初期姿勢（垂直ポーズ）から計算を開始する
    # initial_angles = [0.0] * len(dofbot_chain.links)
   
    # IKの実行（目標ポーズに届く各関節の角度をラジアンで一瞬で計算）
    computed_angles = dofbot_chain.inverse_kinematics_frame(target_frame, initial_position=computed_angles)
    
    # =====================================================================
    # 4. 計算結果を実機用の角度に直して表示
    # =====================================================================
    # print("=========================================")
    # print("          Dofbot IK 計算完了             ")
    # print("=========================================")
    # print(f"目標座標 ➔ X: {target_x*100:.1f}cm, Y: {target_y*100:.1f}cm, Z: {target_z*100:.1f}cm\n")
    
    servo_ids = [1, 2, 3, 4, 5]
    target_angles = {} # 実機命令用の角度を保存する辞書
    
    # index 1〜5（可動する5つのサーボ）の結果を取り出す
    for i, angle_rad in enumerate(computed_angles[1:6]):
        angle_deg = np.degrees(angle_rad)
        
        # 90度を足して、実機サーボ用の命令角度（0度〜180度）に変換
        dofbot_servo_angle = 90.0 + angle_deg
        
        # 小数点第1位までに丸めて保存・表示
        target_angles[servo_ids[i]] = round(dofbot_servo_angle, 1)
        # print(f"サーボ ID {servo_ids[i]} ➔  {target_angles[servo_ids[i]]:.1f} °")
        
    # print("=========================================")
    return target_angles
    
    # Arm.Arm_serial_servo_write6(target_angles[1], target_angles[2], target_angles[3], target_angles[4], target_angles[5], 90, 5000)
    # time.sleep(2)
    
    
    # # =====================================================================
    # # 5. 【参考】実際に実機モータを動かす場合のコード例
    # # =====================================================================
    # # ※ 実際にDofbot（Jetson Nano等）の上で動かす時は、以下のコメントアウトを外します。
    # """
    # from Arm_Lib import Arm_Device
    # import time
    
    # Arm = Arm_Device() # ロボットアームの制御オブジェクトを用意
    
    # # 計算された角度を、すべてのサーボに同時に送信して1秒(1000ms)かけて動かす
    # # Dofbot公式ライブラリの関数の引数は、(ID, 角度, 動かす時間) です。
    # # ※注意：実機の基盤のピン番号（ID）と、ここで計算したIDが一致している必要があります。
    # Arm.Arm_Serial_Servo_Write(1, target_angles[1], 1000)
    # Arm.Arm_Serial_Servo_Write(2, target_angles[2], 1000)
    # Arm.Arm_Serial_Servo_Write(3, target_angles[3], 1000)
    # Arm.Arm_Serial_Servo_Write(4, target_angles[4], 1000)
    # Arm.Arm_Serial_Servo_Write(5, target_angles[5], 1000)
    # time.sleep(1) # 動き終わるまで1秒待つ
    # """

In [17]:
target_x=0
target_y=0
target_z=472.55
angles = cal_servo_angles(target_x,target_y,target_z)
for i in range(5):
    print(angles[i+1])

90.0
90.0
90.0
90.0
90.0


In [22]:
target_x = 0.0
target_y = 0.0
target_z = 0.47255
STEP = 0.01
DURATION = 300    # アームの移動時間（300ミリ秒 = 0.3秒）
WAIT_TIME = 0.3   # Python側の待ち時間（0.3秒）

out_b = widgets.Output(layout=widgets.Layout(height='180px', border='1px solid #cbd5e1', overflow_y='auto'))
label_desc = widgets.HTML("<b>キー監視入力欄：</b> 下のテキストボックスをクリックして W / A / S / D を入力してください。")

key_receiver = widgets.Text(
    value='',
    placeholder='ここにフォーカスを合わせてキー入力',
    layout=widgets.Layout(width='320px')
)

app_b = widgets.VBox([label_desc, key_receiver, out_b])

# ==========================================
# 2. テキストの変更を検知する関数
# ==========================================
def handle_text_change(change):
    global target_x, target_y, target_z
    
    raw_val = change['new']
    if not raw_val:
        return # 自動クリアによる空文字（""）への変化は無視する
        
    key = raw_val[-1].lower()
    
    with out_b:
        # ★ポイント1：前の出力を消去して、常に最新情報だけを1箇所に表示する
        out_b.clear_output(wait=True)
        
        # ★ポイント2：各キーに応じて座標を増減させる
        if key == 'w':
            target_x -= STEP
            print(f"【W】前進（Xを {STEP} 加算）")
        elif key == 's':
            target_x += STEP
            print(f"【S】後退（Xを {STEP} 減算）")
        elif key == 'a':
            target_y -= STEP
            print(f"【A】左スライド（Yを {STEP} 減算）")
        elif key == 'd':
            target_y += STEP
            print(f"【D】右スライド（Yを {STEP} 加算）")
        elif key == 'f':
            target_z += STEP
            print(f"【A】左スライド（Yを {STEP} 減算）")
        elif key == 'r':
            target_z -= STEP
            print(f"【D】右スライド（Yを {STEP} 加算）")
        elif key == 'q':
            print("【Q】セッションを終了しウィジェットを閉じます。")
            app_b.close()
            return
        else:
            print(f"未対応のキーです: {key}")
            # 対応外のキーの場合は、以下の計算をスキップして入力欄をクリアする
            key_receiver.value = ''
            return

        # 座標が更新されたので、サーボの角度を再計算して表示
        try:
            angles = cal_servo_angles(target_x, target_y, target_z)
            print("-" * 30)
            print(f"現在の座標 -> X: {target_x:.2f}, Y: {target_y:.2f}, Z: {target_z:.2f}")
            print("-" * 30)
            print("計算されたサーボ角度:")
            for i in range(5):
                print(f"  サーボ {i+1}: {angles[i+1]} 度")
            Arm.Arm_serial_servo_write6(
                angles[1], angles[2], angles[3], angles[4], angles[5], 90, DURATION
            )
            
            # 4. ★ アームが動き終わるまで待つ（2秒からWAIT_TIME(0.3秒)に変更）
            time.sleep(WAIT_TIME)
        except Exception as e:
            print(f"計算エラーが発生しました: {e}")

    # 入力された値を消去して、次の入力を待ち受ける
    key_receiver.value = ''

# 3. 監視設定と表示
key_receiver.observe(handle_text_change, names='value')
print("【WASD座標制御モード】監視スタート")
display(app_b)

【WASD座標制御モード】監視スタート
